In [1]:
# Bias correct PM2.5

In [2]:
import os
import xarray as xr
import warnings
from utils.utils import adjust_longitude, bilinear_interp, get_scenario_config

In [ ]:
warnings.filterwarnings('ignore')
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "SSP245_G6"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]

PM25_DIR = f"/glade/work/awells/air_quality/{model}/pm25/annual_pm25/"
OBS_DIR = "/glade/work/awells/air_quality/PM2.5_obs/"
SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/pm25_bc/"

# === Main loop ===
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    # Load data arrays
    dates = f"{years.start}-{years.stop}"

    pm25_file = f"PM25_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    pm25_path = os.path.join(PM25_DIR, pm25_file)
    # Convert from kg/m3 to µg/m3
    pm25 = xr.open_dataarray(pm25_path)*10**9
    # Remove the unused lev dimension
    pm25 = pm25.drop_vars("lev")

    hist_file = "PM25_CESM2_hist_01_1990-2010.nc"
    hist_path = os.path.join(PM25_DIR, hist_file)
    # Convert from kg/m3 to µg/m3
    hist = xr.open_dataarray(hist_path)*10**9

    obs_file = "DIMAQ_PM25_1990-2016.nc"
    obs_path = os.path.join(OBS_DIR, obs_file)
    obs = xr.open_dataset(obs_path)["ozone"]

    # Baseline years for fi_2000 and historical
    base = slice("1990", "2010")
    hist_base = hist.sel(year=base).mean("year")
    obs_base = obs.sel(year=base).mean("year")
    obs_base = obs_base.rename({'Longitude': 'lon', 'Latitude': 'lat'})

    # Calculate delta
    delta_fi = adjust_longitude(pm25 / hist_base)

    # Interpolate to the new grid
    regridder = bilinear_interp(delta_fi, obs_base)
    ds_delta_fi = regridder(pm25)

    # Bias correct delta
    bc_pm25 = obs_base * ds_delta_fi

    out_file = f"PM25_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
    out_path = os.path.join(SAVE_DIR, out_file)

    print(f"Saving to {out_path}")
    description = ("Annual mean PM2.5 bias corrected to DIMAQ data "
                   " - scripts by A.F. Wells (2025)")
    bc_pm25.attrs["description"] = description
    bc_pm25.attrs["ensemble_number"] = ens_num
    bc_pm25.attrs["scenario"] = scenario
    bc_pm25.attrs["model"] = model
    bc_pm25.attrs["units"] = "µg/m3"
    bc_pm25.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_01_2035-2069.nc
Processing ARISE, Ensemble 02
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_02_2035-2069.nc
Processing ARISE, Ensemble 03
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_03_2035-2069.nc
Processing ARISE, Ensemble 04
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_04_2035-2069.nc
Processing ARISE, Ensemble 05
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_05_2035-2069.nc
Processing ARISE, Ensemble 06
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_06_2035-2069.nc
Processing ARISE, Ensemble 07
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_07_2035-2069.nc
Processing ARISE, Ensemble 08
Saving to /glade/work/awells/air_quality/CESM/pm25/pm25_bc/PM25_BC_CESM2_ARISE_08_2035-2069.nc
